In [6]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


接下来就可以开发Agent了，基本步骤如下：
Langchain提供了initxxxx
示例代码如下：

In [7]:
# 初始化模型
from langchain.chat_models import init_chat_model
import os
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()


# 这里的最好是从文档里面复制，一般创建key的时候可以下载一个文档，点击下载一下最好
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")



调用init_chat_model, 初始化模型

In [8]:

# 参数 model，model_provider,base_url,api_key
model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key
    
)


print(type(model))  # <class 'langchain_openai.chat_models.base.ChatOpenAI'>


# 除了修改模型提供者以外，init_chat_model方法允许我们调整模型参数，例如：
# - temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
# - max_tokens: 控制生成文本的最大长度
# - top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
# - timeout: 控制生成文本的超时时间
# - max_retries: 控制生成文本的最大重试次数
# - ...

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


2.1.3 使用Model类

初始化model的3种方法，

1. 首先，我们需要安装依赖
  1. LangChain社区依赖：
 uv add langchain-community
  2. 阿里云百炼依赖：
 uv add dashscope
2. 然后，我们就可以使用Model类初始化模型了
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-plus"
    # 其它模型参数...
)
3. 测试，查看生成的模型类型：
print(type(model)) # <class 'langchain_community.chat_models.tongyi.ChatTongyi'>


2.2.1 invoke
invoke方法是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。

In [9]:
# 调用invoke方法
response = model.invoke("月亮的首都是哪里？")

# 查看响应结果
print(response)

2026-11-11 16:12:00 [INFO] httpx2: HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


content='月亮（月球）没有首都。因为月球是地球的天然卫星，而不是一个国家或政治实体，所以不存在“首都”的概念。\n\n如果你对月球的地理环境、科学知识或者人类的探月活动感兴趣，我很乐意为你解答！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 314, 'prompt_tokens': 66, 'total_tokens': 380, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 265, 'rejected_prediction_tokens': None, 'text_tokens': 314}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 66}}, 'model_provider': 'openai', 'model_name': 'qwen3.8-max', 'system_fingerprint': None, 'id': 'chatcmpl-37435b3b-9f80-9eae-b918-f588462ed3cf', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1c9a9-d647-7376-9436-a0af51351243-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 66, 'output_tokens': 314, 'total_tokens': 380, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 265}}


2.2.2 stream
阻塞式调用需要等待较长时间才能看到AI返回的结果，而流式调用则可以实时看到AI返回的一个个词。
示例：

In [12]:
response = model.stream("我感觉我的喜欢有点病态，爱中参杂着很多东西，怎么办？")

# stream调用返回的结果是一个generator，方便我们循环获取结果
print(type(response))

for chunk in response:
    print(chunk.content, end="", flush=True)

<class 'generator'>


2026-11-11 16:28:25 [INFO] httpx2: HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


你会有这种感觉，说明你已经在认真审视自己的内心了——这本身并不“病态”，反而很重要。

很多时候，我们以为的“喜欢/爱”确实不是纯粹的，它可能掺杂着：

- 占有欲、嫉妒、控制欲  
- 依赖感、害怕被抛弃  
- 性吸引、征服欲、新鲜感  
- 想被拯救、想拯救对方  
- 对孤独的逃避  
- 对某种理想形象的投射  
- 过去的创伤、羞耻、愧疚  
- 责任、习惯、利益、现实考量  

这些混在一起，会让人感觉很乱，甚至怀疑自己：“我这样是不是不正常？”  
但人的感情本来就复杂。真正需要关注的，不是“我的感情纯不纯”，而是：**这份感情有没有让你和对方变得更安全、更自由、更尊重彼此？**

你可以试着先问自己几个问题：

1. **我说的“病态”，具体是指什么？**  
   是太强烈？太执着？想控制对方？离不开？还是有很多不该有的念头？

2. **这份喜欢里，哪些是“爱”，哪些是“需要”？**  
   比如：  
   - 爱可能是：我希望对方好，尊重对方的选择。  
   - 需要可能是：我不能没有他/她，我必须被回应，我害怕失去。  

3. **哪些是感受，哪些是行为？**  
   感受可以复杂，但行为要有边界。  
   比如你可以嫉妒、可以不安、可以强烈喜欢，但不能因此跟踪、威胁、羞辱、强迫、伤害自己或对方。

4. **这份感情是在滋养你，还是在消耗你？**  
   如果它让你长期焦虑、失眠、自我厌恶、无法学习工作、反复痛苦，那就需要重视。

5. **我是不是把一些别的东西寄托在对方身上了？**  
   比如安全感、存在感、被看见、被接纳、摆脱孤独、证明自己值得被爱。  
   如果是这样，问题未必是“爱错了”，而是内心有些缺口需要被照顾。

你可以先不急着给自己下结论，比如“我是不是变态”“我是不是有病”。可以换成更温和一点的说法：  
**“我的感情很复杂，我需要慢慢分辨它。”**

如果你愿意，也可以把“爱中参杂着很多东西”具体写出来，比如：  
- 参杂了占有？  
- 参杂了欲望？  
- 参杂了愧疚？  
- 参杂了依赖？  
- 参杂了控制？  
- 参杂了对过去的补偿？  
- 参杂了现实利益？  

写出来之后，你会更容易看清：哪些需要被接纳，哪些需要被约束，哪些需要被疗愈。

如果这种状态已经让你很痛苦，或者你发

In [13]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

# 创建Agent
agent = create_agent(model=model)

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？")
    ]
})

print(response)

2026-11-11 16:52:35 [INFO] httpx2: HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


{'messages': [HumanMessage(content='你好，我是虎哥', additional_kwargs={}, response_metadata={}, id='711f92af-7b81-41b4-85b9-7a7e7059b110'), AIMessage(content='你好，虎哥，很高兴认识你。', additional_kwargs={}, response_metadata={}, id='d23b62ac-6eec-4b09-9047-e4ff59b23f07', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我的名字是什么？', additional_kwargs={}, response_metadata={}, id='97d01d21-e16f-4e08-89bd-aedb629d352a'), AIMessage(content='你的名字是虎哥。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 93, 'total_tokens': 151, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 50, 'rejected_prediction_tokens': None, 'text_tokens': 58}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 93}}, 'model_provider': 'openai', 'model_name': 'qwen3.8-max', 'system_fingerprint': None, 'id': 'chatcmpl-1e418ac

我们可以通过遍历Messages数组，更友好的打印结果：

In [14]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你好，我是虎哥
================================== Ai Message ==================================

你好，虎哥，很高兴认识你。
================================ Human Message =================================

我的名字是什么？
================================== Ai Message ==================================

你的名字是虎哥。


提示：
通过刚才的实现可以发现，拼接message列表可以让AI记住会话历史，产生记忆。不过手动拼接Message太麻烦了，后面我们学习如何实现自动的会话记忆功能。